# 10. MultiIndex & Hierarchical Indexing: Beginner Guide

### 📌 Overview
Master **10. MultiIndex & Hierarchical Indexing: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Construction & Swapping**: Covers `pd.MultiIndex.from_tuples()` and `.swaplevel()`.
- **Multi-Level Slicing**: Covers `.loc[pd.IndexSlice]`.
- **Level Aggregations**: Covers aggregating across index levels.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### 🔹 MultiIndex Construction from Groupby
- **What it does:** Constructs a hierarchical MultiIndex DataFrame grouping by region and card_type.
- **Syntax:** `df.groupby(['region', 'card_type'])[['transaction_amount', 'is_fraud']].mean()`
- **Key Note:** After running `.groupby().agg()`, the grouped columns become index levels. Call `.reset_index()` to bring them back as standard columns.

In [2]:
multi_df = df.groupby(['region', 'card_type'])[['transaction_amount', 'is_fraud']].mean()
print('MultiIndexed Financial Summary:\n', multi_df.round(3))

MultiIndexed Financial Summary:
                     transaction_amount  is_fraud
region  card_type                               
 East   Amex                  1052.769     0.000
        Discover               832.917     0.111
        MasterCard            1147.773     0.095
        Visa                   841.697     0.111
 North  Amex                  1006.639     0.077
        Discover              1169.579     0.154
        MasterCard            1164.634     0.059
        Visa                   914.338     0.000
 South  Amex                  1041.772     0.188
        Discover              1047.690     0.125
        MasterCard            1126.317     0.091
        Visa                  1222.239     0.133
 West   Amex                   818.630     0.040
        Discover              1068.397     0.167
        MasterCard             938.364     0.000
        Visa                   808.629     0.000
East    Amex                  1000.959     0.097
        Discover              1006.0

### 🔹 Swapping MultiIndex Levels with `.swaplevel()`
- **What it does:** Swaps region and card_type index levels.
- **Syntax:** `multi_df.swaplevel(0, 1)`
- **Operation:** `swapped_levels = multi_df.swaplevel('region', 'card_type')`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [3]:
swapped_levels = multi_df.swaplevel('region', 'card_type')
print('Swapped Levels:\n', swapped_levels.head(4))

Swapped Levels:
                    transaction_amount  is_fraud
card_type  region                              
Amex       East           1052.769444  0.000000
Discover   East            832.917059  0.111111
MasterCard East           1147.773000  0.095238
Visa       East            841.696667  0.111111


### 🔹 Cross-Level Slicing with `pd.IndexSlice`
- **What it does:** Extracts all metrics for 'Visa' cards across all global regions.
- **Syntax:** `multi_df.loc[pd.IndexSlice[:, 'Visa'], :]`
- **Operation:** `idx = pd.IndexSlice`
- **Key Note:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.

In [4]:
idx = pd.IndexSlice
visa_slice = multi_df.loc[idx[:, 'Visa'], :]
print('Visa Slices Across All Regions:\n', visa_slice)

Visa Slices Across All Regions:
                    transaction_amount  is_fraud
region  card_type                              
 East   Visa               841.696667  0.111111
 North  Visa               914.338235  0.000000
 South  Visa              1222.239333  0.133333
 West   Visa               808.628571  0.000000
East    Visa               969.947901  0.094668
North   Visa              1013.205530  0.125000
South   Visa              1011.523776  0.110988
West    Visa              1002.677195  0.113250
east    Visa               954.759000  0.150000
north   Visa              1321.958750  0.187500
south   Visa              1014.284783  0.130435
west    Visa              1054.587059  0.052632


### 🔹 Aggregating Across MultiIndex Levels
- **What it does:** Computes top-level region totals collapsing card_type level.
- **Syntax:** `multi_df.groupby(level='region').mean()`
- **Operation:** `region_collapsed = multi_df.groupby(level='region').mean()`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [5]:
region_collapsed = multi_df.groupby(level='region').mean()
print('Collapsed Region Level Averages:\n', region_collapsed)

Collapsed Region Level Averages:
          transaction_amount  is_fraud
region                               
 East            968.789042  0.079365
 North          1063.797596  0.072398
 South          1109.504620  0.134186
 West            908.505029  0.051667
East             994.671452  0.098900
North           1002.053251  0.113773
South           1006.220312  0.105276
West            1008.967594  0.112307
east            1104.690242  0.159545
north           1083.383611  0.180351
south           1087.525212  0.099444
west            1120.761021  0.125279


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Cross-Sectional MultiIndex Lookup with `.xs()`
- **Objective:** Q1: Cross-Sectional MultiIndex Lookup with `.xs()`
- **Approach:** Extract all regional statistics for 'MasterCard' using `.xs()` without resetting index.
- **Syntax:** `multi_df.xs('MasterCard', level='card_type')`

In [6]:
mc_xs = multi_df.xs('MasterCard', level='card_type')
print('MasterCard Cross-Section (.xs):\n', mc_xs)

MasterCard Cross-Section (.xs):
          transaction_amount  is_fraud
region                               
 East           1147.773000  0.095238
 North          1164.634118  0.058824
 South          1126.317273  0.090909
 West            938.364444  0.000000
East            1001.756636  0.101402
North            994.791926  0.103175
South           1008.329345  0.104839
West             986.606820  0.111111
east            1254.708636  0.318182
north           1033.943750  0.235294
south           1171.810000  0.041667
west             909.845500  0.000000
